# Notebook Overview — Evaluate Development Results

## Purpose

This notebook evaluates VideoQA experiment results and generates performance metrics, verification summaries, analysis tables, visualizations, and reporting artifacts.

The workflow loads VideoQA prediction results, experiment summaries, runtime statistics, evidence metadata, representation metadata, and NExT-QA annotation data produced by previous notebooks and performs quantitative evaluation of VideoQA performance.

During development, this notebook evaluates baseline VideoQA outputs generated by Notebook 02. The notebook is designed to be extended to support comparative evaluation of baseline, pretrained-representation, and autoencoder-representation experiments after full-dataset experiments are introduced.

Evaluation procedures include prediction verification, answer-quality assessment, reasoning-category analysis, runtime analysis, representation-effectiveness analysis, and comparative reporting across experimental workflows.

## Inputs

* VideoQA prediction results
* Experiment summary reports
* Runtime statistics
* Evidence metadata
* Representation metadata (when available)
* NExT-QA question annotations
* Project configuration settings

## Outputs

* Evaluation metrics tables
* Prediction verification summaries
* Category performance analyses
* Runtime analyses
* Representation analyses
* Performance visualizations
* Evaluation reports
* Saved reporting artifacts

## Workflow

The workflow begins by loading experiment outputs and reference annotation data. Inputs are validated before prediction quality is verified and evaluation metrics are computed.

Evaluation results are analyzed across reasoning categories, question types, runtime characteristics, and representation-learning workflows. Visualization and reporting artifacts are generated to support experiment comparison and project documentation.

The notebook is designed to support comparative evaluation of baseline, pretrained-representation, and autoencoder-representation VideoQA experiments using a common evaluation framework.


### 🔷 Step 1 — Clone Required Repository Files

* Clone the project repository using sparse checkout to minimize download size and runtime initialization overhead.
* Authenticate access to the private GitHub repository using a fine-grained access token stored in Google Colab Secrets.
* Configure the local notebook workspace and change to the repository working directory.
* Verify that required repository files and directories are available for subsequent notebook execution.
* Optionally display repository paths, directory contents, and cloned files when `VERBOSE=True`.


In [ ]:
# ============================================================
# Step 1: Clone Required Repository Files
# ============================================================

# ============================================================
# Runtime Settings
# ============================================================
VERBOSE = True
REQUIRE_L4_GPU = True

import os
from google.colab import userdata

REPO_NAME = "videoqa-representation-comparison"
REPO_OWNER = "pgailinas"
REPO_BASE_DIR = "/content"
REPO_DIR = os.path.join(REPO_BASE_DIR, REPO_NAME)

# ------------------------------------------------------------
# Retrieve GitHub Token from Colab Secrets
# ------------------------------------------------------------

github_token = userdata.get("GITHUB_TOKEN")

if github_token is None:
    raise ValueError(
        "GITHUB_TOKEN not found in Colab Secrets."
    )

repo_url = (
    f"https://{github_token}"
    f"@github.com/{REPO_OWNER}/{REPO_NAME}.git"
)

# ------------------------------------------------------------
# Move to Base Directory
# ------------------------------------------------------------

%cd {REPO_BASE_DIR}

# ------------------------------------------------------------
# Clone Repository if Needed
# ------------------------------------------------------------

if not os.path.exists(REPO_DIR):

    if VERBOSE:
        print("Cloning required repository directories...")

    !git clone --quiet --filter=blob:none --no-checkout {repo_url}

    %cd {REPO_DIR}

    !git sparse-checkout init --cone

    !git sparse-checkout set \
        src \
        datasets \
        outputs

    !git checkout --quiet main

else:

    if VERBOSE:
        print(f"Repository already exists: {REPO_DIR}")

    %cd {REPO_DIR}

# ------------------------------------------------------------
# Verify Repository Setup
# ------------------------------------------------------------

required_paths = [
    "src",
    "datasets",
    "outputs",
    "datasets/NExT-QA",
    "datasets/NExT-QA/questions",
    "datasets/NExT-QA/metadata",
    "src/videoqa_representation_config.py",
    "src/nextqa_video_cache.py",
    "src/nextqa_metadata.py",
    "src/video_evidence.py",
    "src/evidence_validation.py",
    "src/evidence_io.py",
]

for path in required_paths:

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Required path not found: {path}"
        )

print("Repository setup complete.")

if VERBOSE:
    print(f"\nCurrent directory: {os.getcwd()}")
    print("\nRepository directories:")
    !find src datasets outputs \
        -maxdepth 2 \
        -type d \
        ! -path "*/__pycache__*" | sort



### 🔷 Step 2 — Load Evaluation Data

* Load VideoQA prediction results generated by previous experiment notebooks.
* Load experiment summary statistics and runtime metrics.
* Load evidence metadata generated by Notebook 02.
* Load representation metadata when available.
* Load NExT-QA annotation records required for evaluation and reporting.
* Display dataset sizes and file statistics to verify successful loading.



In [ ]:
# ============================================================
# Step 2: Load Evaluation Data
# ============================================================

import pandas as pd

from src.videoqa_representation_config import *

print("Loading evaluation data...\n")

# ------------------------------------------------------------
# Input Files
# ------------------------------------------------------------

VALIDATION_ANNOTATIONS_CSV = (
    QUESTIONS_DIR / "val.csv"
)

required_input_files = [
    BASELINE_PREDICTIONS_CSV,
    BASELINE_SUMMARY_CSV,
    EVIDENCE_METADATA_CSV,
    EVIDENCE_SUMMARY_CSV,
    VALIDATION_ANNOTATIONS_CSV,
]

missing_input_files = [
    file_path
    for file_path in required_input_files
    if not file_path.exists()
]

if missing_input_files:
    for file_path in missing_input_files:
        print(f"Missing required input file: {file_path}")

    raise FileNotFoundError(
        "One or more required evaluation input files are missing."
    )

# ------------------------------------------------------------
# Load Baseline Experiment Results
# ------------------------------------------------------------

baseline_predictions_df = pd.read_csv(
    BASELINE_PREDICTIONS_CSV
)

baseline_summary_df = pd.read_csv(
    BASELINE_SUMMARY_CSV
)

# ------------------------------------------------------------
# Load Evaluation Reference Data
# ------------------------------------------------------------

val_annotations_df = pd.read_csv(
    VALIDATION_ANNOTATIONS_CSV
)

evidence_metadata_df = pd.read_csv(
    EVIDENCE_METADATA_CSV
)

evidence_summary_df = pd.read_csv(
    EVIDENCE_SUMMARY_CSV
)

# ------------------------------------------------------------
# Display Dataset Information
# ------------------------------------------------------------

print("Loaded Evaluation Data")
print("-" * 60)
print(
    f"Baseline Predictions      : "
    f"{len(baseline_predictions_df):,} records"
)
print(
    f"Baseline Summary          : "
    f"{len(baseline_summary_df):,} records"
)
print(
    f"Validation Annotations    : "
    f"{len(val_annotations_df):,} records"
)
print(
    f"Evidence Metadata         : "
    f"{len(evidence_metadata_df):,} records"
)
print(
    f"Evidence Summary          : "
    f"{len(evidence_summary_df):,} records"
)

print("\nInput Files")
print("-" * 60)
for file_path in required_input_files:
    print(file_path)

# ------------------------------------------------------------
# Display Available Columns
# ------------------------------------------------------------

print("\nBaseline Prediction Columns")
print("-" * 60)
print(list(baseline_predictions_df.columns))

print("\nBaseline Summary Columns")
print("-" * 60)
print(list(baseline_summary_df.columns))

# ------------------------------------------------------------
# Preview Loaded Data
# ------------------------------------------------------------

print("\nBaseline Predictions Preview")
display(baseline_predictions_df.head())

print("\nBaseline Summary Preview")
display(baseline_summary_df.head())



### 🔷 Step 3 — Validate Evaluation Inputs

* Verify that all required evaluation input files are present.
* Confirm required columns exist in prediction and summary datasets.
* Validate record counts and data integrity.
* Check for missing values, duplicate records, and invalid entries.
* Report validation results before evaluation processing begins.


In [ ]:
# ============================================================
# Step 3: Validate Evaluation Inputs
# ============================================================

print("Validating evaluation inputs...\n")

validation_passed = True

# ------------------------------------------------------------
# Verify Required DataFrames
# ------------------------------------------------------------
required_dataframes = {
    "Baseline Predictions": baseline_predictions_df,
    "Baseline Summary": baseline_summary_df,
    "Validation Annotations": val_annotations_df,
    "Evidence Metadata": evidence_metadata_df,
    "Evidence Summary": evidence_summary_df,
}

print("Dataset Validation")
print("-" * 60)

for name, df in required_dataframes.items():
    record_count = len(df)
    print(
        f"{name:<25}: "
        f"{record_count:>8,} records"
    )
    if record_count == 0:
        validation_passed = False
        print(f"  ERROR: {name} contains no records")

# ------------------------------------------------------------
# Verify Required Prediction Columns
# ------------------------------------------------------------
required_prediction_columns = [
    "video",
    "question",
    "ground_truth",
    "prediction",
    "evidence_record_count",
]

missing_prediction_columns = [
    column
    for column in required_prediction_columns
    if column not in baseline_predictions_df.columns
]

# ------------------------------------------------------------
# Verify Required Summary Columns
# ------------------------------------------------------------
required_summary_columns = [
    "metric",
    "value",
]

missing_summary_columns = [
    column
    for column in required_summary_columns
    if column not in baseline_summary_df.columns
]

# ------------------------------------------------------------
# Report Missing Columns
# ------------------------------------------------------------
print("\nColumn Validation")
print("-" * 60)

if missing_prediction_columns:
    validation_passed = False
    print(
        "Missing Prediction Columns:"
    )
    for column in missing_prediction_columns:
        print(f"  {column}")
else:
    print(
        "Prediction columns validated."
    )

if missing_summary_columns:
    validation_passed = False
    print(
        "Missing Summary Columns:"
    )
    for column in missing_summary_columns:
        print(f"  {column}")
else:
    print(
        "Summary columns validated."
    )

# ------------------------------------------------------------
# Verify Summary Metrics
# ------------------------------------------------------------
required_metrics = [
    "total_predictions",
    "valid_predictions",
    "missing_predictions",
    "empty_predictions",
    "error_predictions",
    "unique_videos",
    "elapsed_time_seconds",
]

available_metrics = set(
    baseline_summary_df["metric"]
)

missing_metrics = [
    metric
    for metric in required_metrics
    if metric not in available_metrics
]

print("\nMetric Validation")
print("-" * 60)

if missing_metrics:
    validation_passed = False
    print(
        "Missing Summary Metrics:"
    )
    for metric in missing_metrics:
        print(f"  {metric}")
else:
    print(
        "Summary metrics validated."
    )

# ------------------------------------------------------------
# Final Validation Status
# ------------------------------------------------------------

print("\nValidation Results")
print("-" * 60)

print(
    f"Validation Passed : "
    f"{validation_passed}"
)

if not validation_passed:
    raise ValueError(
        "Evaluation input validation failed."
    )



### 🔷 Step 4 — Prepare Evaluation Dataset

* Create a unified evaluation dataset from VideoQA prediction results and NExT-QA annotation metadata.
* Standardize prediction, question, and answer fields for evaluation processing.
* Attach question-type information and reasoning-category assignments.
* Generate unique evaluation identifiers for each prediction record.
* Summarize evaluation records, category distributions, and question-type distributions.
* Prepare the evaluation dataset used by all subsequent evaluation procedures.




In [ ]:
# ============================================================
# Step 4: Prepare Evaluation Dataset
# ============================================================

print("Preparing evaluation dataset...\n")

# ------------------------------------------------------------
# Copy Baseline Predictions
# ------------------------------------------------------------

evaluation_df = baseline_predictions_df.copy()

# ------------------------------------------------------------
# Add Experiment Metadata
# ------------------------------------------------------------

evaluation_df["experiment"] = "baseline"
evaluation_df["model"] = "Qwen2-VL-7B"

# ------------------------------------------------------------
# Standardize Key Fields
# ------------------------------------------------------------

evaluation_df["video"] = evaluation_df["video"].astype(str)
evaluation_df["question"] = evaluation_df["question"].astype(str)
evaluation_df["ground_truth"] = evaluation_df["ground_truth"].astype(str)
evaluation_df["prediction"] = evaluation_df["prediction"].astype(str)

# ------------------------------------------------------------
# Attach NExT-QA Annotation Metadata When Available
# ------------------------------------------------------------

annotation_metadata_columns = [
    column
    for column in [
        "video",
        "question",
        "answer",
        "type",
        "qid",
        "question_id",
    ]
    if column in val_annotations_df.columns
]

annotation_metadata_df = val_annotations_df[
    annotation_metadata_columns
].copy()

annotation_metadata_df["video"] = (
    annotation_metadata_df["video"].astype(str)
)
annotation_metadata_df["question"] = (
    annotation_metadata_df["question"].astype(str)
)

# Avoid duplicating ground-truth answer if prediction file already has it
if "answer" in annotation_metadata_df.columns:
    annotation_metadata_df = annotation_metadata_df.rename(
        columns={"answer": "annotation_answer"}
    )

evaluation_df = evaluation_df.merge(
    annotation_metadata_df,
    on=["video", "question"],
    how="left"
)

# ------------------------------------------------------------
# Derive Reasoning Category from NExT-QA Question Type
# ------------------------------------------------------------

def map_reasoning_category(question_type):
    if pd.isna(question_type):
        return "Unknown"

    question_type = str(question_type).upper()

    if question_type.startswith("C"):
        return "Causal"
    if question_type.startswith("T"):
        return "Temporal"
    if question_type.startswith("D"):
        return "Descriptive"

    return "Unknown"


if "type" in evaluation_df.columns:
    evaluation_df["reasoning_category"] = (
        evaluation_df["type"].apply(map_reasoning_category)
    )
else:
    evaluation_df["type"] = "Unknown"
    evaluation_df["reasoning_category"] = "Unknown"

# ------------------------------------------------------------
# Add Evaluation Record Identifier
# ------------------------------------------------------------

evaluation_df.insert(
    0,
    "evaluation_id",
    [
        f"eval_{index:06d}"
        for index in range(len(evaluation_df))
    ]
)

# ------------------------------------------------------------
# Summarize Evaluation Dataset
# ------------------------------------------------------------

matched_annotations = (
    evaluation_df["annotation_answer"].notna().sum()
    if "annotation_answer" in evaluation_df.columns
    else 0
)

print("Evaluation Dataset Prepared")
print("-" * 60)
print(f"Evaluation Records       : {len(evaluation_df):,}")
print(f"Matched Annotations      : {matched_annotations:,}")
print(f"Unique Videos            : {evaluation_df['video'].nunique():,}")
print(f"Experiments              : {evaluation_df['experiment'].nunique():,}")

print("\nReasoning Category Counts")
print("-" * 60)
display(
    evaluation_df["reasoning_category"]
    .value_counts()
    .rename_axis("reasoning_category")
    .reset_index(name="count")
)

print("\nQuestion Type Counts")
print("-" * 60)
display(
    evaluation_df["type"]
    .value_counts()
    .rename_axis("question_type")
    .reset_index(name="count")
)

print("\nEvaluation Dataset Preview")
display(evaluation_df.head())



### 🔷 Step 5 — Verify Prediction Quality

* Validate prediction completeness and identify missing or invalid responses.
* Normalize ground-truth answers and model predictions for comparison.
* Detect exact-match and partial-match answer agreements.
* Measure prediction validity, coverage, and match rates.
* Generate prediction verification summaries and quality statistics.
* Prepare verified prediction results for evaluation metric computation.



In [ ]:
# ============================================================
# Step 5: Verify Prediction Quality
# ============================================================

import re

print("Verifying prediction quality...\n")

# ------------------------------------------------------------
# Text Normalization Helper
# ------------------------------------------------------------

def normalize_answer_text(text):
    """Normalize answer text for simple exact-match comparison."""
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ------------------------------------------------------------
# Normalize Ground Truth and Predictions
# ------------------------------------------------------------

evaluation_df["ground_truth_normalized"] = (
    evaluation_df["ground_truth"].apply(normalize_answer_text)
)

evaluation_df["prediction_normalized"] = (
    evaluation_df["prediction"].apply(normalize_answer_text)
)

# ------------------------------------------------------------
# Verify Prediction Status
# ------------------------------------------------------------

evaluation_df["prediction_missing"] = (
    evaluation_df["prediction"].isna()
    | (evaluation_df["prediction"].astype(str).str.strip() == "")
)

evaluation_df["prediction_error"] = (
    evaluation_df["prediction"]
    .astype(str)
    .str.lower()
    .str.contains("error|cuda|out of memory|failed", regex=True)
)

evaluation_df["prediction_valid"] = (
    ~evaluation_df["prediction_missing"]
    & ~evaluation_df["prediction_error"]
)

# ------------------------------------------------------------
# Compare Predictions with Ground Truth
# ------------------------------------------------------------

evaluation_df["exact_match"] = (
    evaluation_df["prediction_normalized"]
    == evaluation_df["ground_truth_normalized"]
)

evaluation_df["ground_truth_contains_prediction"] = (
    evaluation_df.apply(
        lambda row: (
            row["prediction_normalized"] in row["ground_truth_normalized"]
            and row["prediction_normalized"] != ""
        ),
        axis=1
    )
)

evaluation_df["prediction_contains_ground_truth"] = (
    evaluation_df.apply(
        lambda row: (
            row["ground_truth_normalized"] in row["prediction_normalized"]
            and row["ground_truth_normalized"] != ""
        ),
        axis=1
    )
)

evaluation_df["partial_match"] = (
    evaluation_df["ground_truth_contains_prediction"]
    | evaluation_df["prediction_contains_ground_truth"]
)

# ------------------------------------------------------------
# Build Prediction Verification Summary
# ------------------------------------------------------------

total_predictions = len(evaluation_df)
valid_predictions = int(evaluation_df["prediction_valid"].sum())
missing_predictions = int(evaluation_df["prediction_missing"].sum())
error_predictions = int(evaluation_df["prediction_error"].sum())
exact_matches = int(evaluation_df["exact_match"].sum())
partial_matches = int(evaluation_df["partial_match"].sum())

prediction_verification_df = pd.DataFrame(
    [
        {
            "metric": "total_predictions",
            "value": total_predictions,
        },
        {
            "metric": "valid_predictions",
            "value": valid_predictions,
        },
        {
            "metric": "missing_predictions",
            "value": missing_predictions,
        },
        {
            "metric": "error_predictions",
            "value": error_predictions,
        },
        {
            "metric": "exact_matches",
            "value": exact_matches,
        },
        {
            "metric": "partial_matches",
            "value": partial_matches,
        },
        {
            "metric": "exact_match_rate",
            "value": (
                exact_matches / total_predictions
                if total_predictions > 0
                else 0
            ),
        },
        {
            "metric": "partial_match_rate",
            "value": (
                partial_matches / total_predictions
                if total_predictions > 0
                else 0
            ),
        },
        {
            "metric": "valid_prediction_rate",
            "value": (
                valid_predictions / total_predictions
                if total_predictions > 0
                else 0
            ),
        },
    ]
)

# ------------------------------------------------------------
# Display Verification Results
# ------------------------------------------------------------

print("Prediction Quality Verification")
print("-" * 60)
print(f"Total Predictions       : {total_predictions:,}")
print(f"Valid Predictions       : {valid_predictions:,}")
print(f"Missing Predictions     : {missing_predictions:,}")
print(f"Error Predictions       : {error_predictions:,}")
print(f"Exact Matches           : {exact_matches:,}")
print(f"Partial Matches         : {partial_matches:,}")
print(f"Exact Match Rate        : {exact_matches / total_predictions:.2%}")
print(f"Partial Match Rate      : {partial_matches / total_predictions:.2%}")

print("\nPrediction Verification Summary")
print("-" * 60)
display(prediction_verification_df)

print("\nSample Verification Records")
print("-" * 60)
display(
    evaluation_df[
        [
            "evaluation_id",
            "video",
            "question",
            "ground_truth",
            "prediction",
            "exact_match",
            "partial_match",
            "prediction_valid",
        ]
    ].head()
)



### 🔷 Step 6 — Compute Evaluation Metrics

* Calculate overall evaluation metrics from verified prediction results.
* Measure exact-match accuracy, partial-match accuracy, and prediction coverage.
* Generate performance summaries by reasoning category.
* Generate performance summaries by NExT-QA question type.
* Analyze ground-truth and prediction answer lengths.
* Produce evaluation metrics used for comparative VideoQA analysis.

**Note:**
Exact-match and substring-based metrics provide a baseline evaluation framework but may underestimate VideoQA answer quality due to paraphrasing and semantic equivalence. Future work may incorporate semantic similarity metrics to better evaluate semantically equivalent answers expressed using different wording.



In [ ]:
# ============================================================
# Step 6: Compute Evaluation Metrics
# ============================================================

print("Computing evaluation metrics...\n")

# ------------------------------------------------------------
# Overall Evaluation Metrics
# ------------------------------------------------------------

total_records = len(evaluation_df)
valid_records = int(evaluation_df["prediction_valid"].sum())
exact_matches = int(evaluation_df["exact_match"].sum())
partial_matches = int(evaluation_df["partial_match"].sum())

overall_metrics = [
    {
        "metric": "total_evaluation_records",
        "value": total_records,
    },
    {
        "metric": "valid_predictions",
        "value": valid_records,
    },
    {
        "metric": "exact_matches",
        "value": exact_matches,
    },
    {
        "metric": "partial_matches",
        "value": partial_matches,
    },
    {
        "metric": "exact_match_accuracy",
        "value": exact_matches / total_records if total_records > 0 else 0,
    },
    {
        "metric": "partial_match_accuracy",
        "value": partial_matches / total_records if total_records > 0 else 0,
    },
    {
        "metric": "valid_prediction_rate",
        "value": valid_records / total_records if total_records > 0 else 0,
    },
]

evaluation_metrics_df = pd.DataFrame(overall_metrics)

# ------------------------------------------------------------
# Metrics by Reasoning Category
# ------------------------------------------------------------

category_metrics_df = (
    evaluation_df
    .groupby("reasoning_category", dropna=False)
    .agg(
        total_records=("evaluation_id", "count"),
        valid_predictions=("prediction_valid", "sum"),
        exact_matches=("exact_match", "sum"),
        partial_matches=("partial_match", "sum"),
    )
    .reset_index()
)

category_metrics_df["exact_match_accuracy"] = (
    category_metrics_df["exact_matches"]
    / category_metrics_df["total_records"]
)

category_metrics_df["partial_match_accuracy"] = (
    category_metrics_df["partial_matches"]
    / category_metrics_df["total_records"]
)

category_metrics_df["valid_prediction_rate"] = (
    category_metrics_df["valid_predictions"]
    / category_metrics_df["total_records"]
)

# ------------------------------------------------------------
# Metrics by Question Type
# ------------------------------------------------------------

question_type_metrics_df = (
    evaluation_df
    .groupby("type", dropna=False)
    .agg(
        total_records=("evaluation_id", "count"),
        valid_predictions=("prediction_valid", "sum"),
        exact_matches=("exact_match", "sum"),
        partial_matches=("partial_match", "sum"),
    )
    .reset_index()
    .rename(columns={"type": "question_type"})
)

question_type_metrics_df["exact_match_accuracy"] = (
    question_type_metrics_df["exact_matches"]
    / question_type_metrics_df["total_records"]
)

question_type_metrics_df["partial_match_accuracy"] = (
    question_type_metrics_df["partial_matches"]
    / question_type_metrics_df["total_records"]
)

question_type_metrics_df["valid_prediction_rate"] = (
    question_type_metrics_df["valid_predictions"]
    / question_type_metrics_df["total_records"]
)

# ------------------------------------------------------------
# Answer Length Metrics
# ------------------------------------------------------------

evaluation_df["ground_truth_word_count"] = (
    evaluation_df["ground_truth_normalized"]
    .str.split()
    .apply(len)
)

evaluation_df["prediction_word_count"] = (
    evaluation_df["prediction_normalized"]
    .str.split()
    .apply(len)
)

answer_length_metrics_df = pd.DataFrame(
    [
        {
            "metric": "average_ground_truth_word_count",
            "value": evaluation_df["ground_truth_word_count"].mean(),
        },
        {
            "metric": "median_ground_truth_word_count",
            "value": evaluation_df["ground_truth_word_count"].median(),
        },
        {
            "metric": "average_prediction_word_count",
            "value": evaluation_df["prediction_word_count"].mean(),
        },
        {
            "metric": "median_prediction_word_count",
            "value": evaluation_df["prediction_word_count"].median(),
        },
        {
            "metric": "minimum_prediction_word_count",
            "value": evaluation_df["prediction_word_count"].min(),
        },
        {
            "metric": "maximum_prediction_word_count",
            "value": evaluation_df["prediction_word_count"].max(),
        },
    ]
)

# ------------------------------------------------------------
# Display Evaluation Metrics
# ------------------------------------------------------------

print("Overall Evaluation Metrics")
print("-" * 60)
display(evaluation_metrics_df)

print("\nMetrics by Reasoning Category")
print("-" * 60)
display(category_metrics_df)

print("\nMetrics by Question Type")
print("-" * 60)
display(question_type_metrics_df)

print("\nAnswer Length Metrics")
print("-" * 60)
display(answer_length_metrics_df)



### 🔷 Step 7 — Generate Runtime Analysis

* Analyze VideoQA evaluation execution performance.
* Summarize total runtime and average inference time per sample.
* Estimate processing requirements for larger evaluation workloads.
* Calculate projected runtimes for validation-split and full-dataset execution.
* Generate runtime analysis summaries for evaluation reporting.



In [ ]:
# ============================================================
# Step 7: Generate Runtime Analysis
# ============================================================

print("Generating runtime analysis...\n")

# ------------------------------------------------------------
# Helper Function for Summary Metric Lookup
# ------------------------------------------------------------

def get_metric(metric_name, default=None):
    metric_rows = baseline_summary_df[
        baseline_summary_df["metric"] == metric_name
    ]

    if metric_rows.empty:
        return default

    return metric_rows["value"].iloc[0]

# ------------------------------------------------------------
# Extract Runtime Metrics
# ------------------------------------------------------------

runtime_analysis = {
    "elapsed_time_seconds":
        get_metric("elapsed_time_seconds"),

    "average_time_per_sample_seconds":
        get_metric("average_time_per_sample_seconds"),

    "projected_validation_runtime_minutes":
        get_metric("projected_validation_runtime_minutes"),

    "projected_full_dataset_runtime_hours":
        get_metric("projected_full_dataset_runtime_hours"),

    "total_predictions":
        get_metric("total_predictions"),

    "valid_predictions":
        get_metric("valid_predictions"),
}

runtime_analysis_df = pd.DataFrame(
    list(runtime_analysis.items()),
    columns=["metric", "value"]
)

# ------------------------------------------------------------
# Add Human-Readable Runtime Values
# ------------------------------------------------------------

elapsed_seconds = float(
    runtime_analysis["elapsed_time_seconds"]
)

average_seconds = float(
    runtime_analysis["average_time_per_sample_seconds"]
)

validation_minutes = float(
    runtime_analysis["projected_validation_runtime_minutes"]
)

full_dataset_hours = float(
    runtime_analysis["projected_full_dataset_runtime_hours"]
)

runtime_summary_df = pd.DataFrame(
    [
        {
            "runtime_metric": "Elapsed runtime",
            "value": elapsed_seconds,
            "unit": "seconds",
        },
        {
            "runtime_metric": "Average runtime per sample",
            "value": average_seconds,
            "unit": "seconds/sample",
        },
        {
            "runtime_metric": "Projected validation split runtime",
            "value": validation_minutes,
            "unit": "minutes",
        },
        {
            "runtime_metric": "Projected full dataset runtime",
            "value": full_dataset_hours,
            "unit": "hours",
        },
    ]
)

# ------------------------------------------------------------
# Display Runtime Analysis
# ------------------------------------------------------------

print("Runtime Analysis Summary")
print("-" * 60)

display(runtime_summary_df)

print("\nRuntime Interpretation")
print("-" * 60)
print(
    f"The baseline run processed "
    f"{int(runtime_analysis['valid_predictions']):,} valid samples "
    f"in {elapsed_seconds:.2f} seconds."
)

print(
    f"Average runtime was "
    f"{average_seconds:.2f} seconds per sample."
)

print(
    f"Projected runtime for the full validation split is "
    f"{validation_minutes:.2f} minutes."
)

print(
    f"Projected runtime for the full dataset is "
    f"{full_dataset_hours:.2f} hours."
)



### 🔷 Step 8 — Generate Evidence and Representation Analysis

* Analyze evidence records used during VideoQA inference.
* Summarize evidence utilization across evaluated samples.
* Compare evidence usage against available repository evidence.
* Analyze representation characteristics when representation metadata is available.
* Calculate evidence and representation summary statistics.
* Generate analysis summaries for evaluation reporting and visualization.



In [ ]:
# ============================================================
# Step 8: Generate Evidence Utilization Analysis
# ============================================================

print("Generating evidence utilization analysis...\n")

# ------------------------------------------------------------
# Analyze Evidence Records Used During Baseline Inference
# ------------------------------------------------------------

evidence_usage_summary = {
    "samples_analyzed":
        len(baseline_predictions_df),

    "total_evidence_records_used":
        baseline_predictions_df["evidence_record_count"].sum(),

    "average_evidence_records_per_sample":
        baseline_predictions_df["evidence_record_count"].mean(),

    "minimum_evidence_records_per_sample":
        baseline_predictions_df["evidence_record_count"].min(),

    "maximum_evidence_records_per_sample":
        baseline_predictions_df["evidence_record_count"].max(),

    "median_evidence_records_per_sample":
        baseline_predictions_df["evidence_record_count"].median(),
}

evidence_usage_df = pd.DataFrame(
    list(evidence_usage_summary.items()),
    columns=["metric", "value"]
)

# ------------------------------------------------------------
# Analyze Available Evidence Metadata
# ------------------------------------------------------------

available_evidence_summary = {
    "total_available_evidence_records":
        len(evidence_metadata_df),

    "unique_videos_with_evidence":
        evidence_metadata_df["video_id"].nunique(),

    "average_available_evidence_per_video":
        (
            evidence_metadata_df
            .groupby("video_id")
            .size()
            .mean()
        ),

    "minimum_available_evidence_per_video":
        (
            evidence_metadata_df
            .groupby("video_id")
            .size()
            .min()
        ),

    "maximum_available_evidence_per_video":
        (
            evidence_metadata_df
            .groupby("video_id")
            .size()
            .max()
        ),
}

available_evidence_df = pd.DataFrame(
    list(available_evidence_summary.items()),
    columns=["metric", "value"]
)

available_evidence_df["value"] = (
    available_evidence_df["value"]
    .apply(
        lambda x:
        round(x, 2)
        if isinstance(x, (int, float))
        else x
    )
)

# ------------------------------------------------------------
# Display Evidence Utilization Analysis
# ------------------------------------------------------------

print("Evidence Records Used During Baseline Inference")
print("-" * 60)
display(evidence_usage_df)

print("\nAvailable Evidence Repository Summary")
print("-" * 60)
display(available_evidence_df)

# ------------------------------------------------------------
# Display Evidence Record Count Distribution
# ------------------------------------------------------------

print("\nEvidence Records Per Evaluated Sample")
print("-" * 60)

display(
    baseline_predictions_df[
        [
            "video",
            "question",
            "evidence_record_count",
        ]
    ].sort_values(
        by="evidence_record_count",
        ascending=False
    ).head(10)
)



### 🔷 Step 9 — Create Visualizations

* Generate visualizations summarizing VideoQA evaluation results.
* Visualize evaluation metrics, reasoning-category performance, runtime analysis, evidence utilization statistics, and representation-learning results.
* Compare ground-truth and predicted answer characteristics.
* Create publication-ready figures suitable for reports and presentations.
* Display generated visualizations within the notebook and save figure files for export.

In [ ]:
# ============================================================
# Step 9: Create Visualizations
# ============================================================

import os
import matplotlib.pyplot as plt

print("Creating visualizations...\n")

# ------------------------------------------------------------
# Create Output Directory
# ------------------------------------------------------------

figure_dir = "outputs/evaluation/figures"

os.makedirs(
    figure_dir,
    exist_ok=True
)

generated_figures = []


# ------------------------------------------------------------
# Helper Function
# ------------------------------------------------------------

def save_current_figure(filename):
    figure_path = os.path.join(
        figure_dir,
        filename
    )

    plt.tight_layout()
    plt.savefig(
        figure_path,
        dpi=150,
        bbox_inches="tight"
    )
    plt.show()
    plt.close()

    generated_figures.append(
        {
            "figure": filename,
            "size_kb": os.path.getsize(figure_path) / 1024,
        }
    )


# ------------------------------------------------------------
# Visualization 1: Evaluation Metrics Summary
# ------------------------------------------------------------

metric_plot_df = evaluation_metrics_df[
    evaluation_metrics_df["metric"].isin(
        [
            "valid_prediction_rate",
            "partial_match_accuracy",
            "exact_match_accuracy",
        ]
    )
].copy()

metric_plot_df["metric_label"] = metric_plot_df["metric"].map(
    {
        "valid_prediction_rate": "Valid Prediction Rate",
        "partial_match_accuracy": "Partial Match Accuracy",
        "exact_match_accuracy": "Exact Match Accuracy",
    }
)

metric_plot_df["percentage"] = metric_plot_df["value"] * 100

plt.figure(figsize=(8, 5))
plt.bar(
    metric_plot_df["metric_label"],
    metric_plot_df["percentage"]
)
plt.ylabel("Percentage")
plt.title("Baseline Evaluation Metrics")
plt.xticks(rotation=25, ha="right")
plt.ylim(0, 100)

save_current_figure("evaluation_metrics_summary.png")


# ------------------------------------------------------------
# Visualization 2: Reasoning Category Performance
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))
plt.bar(
    category_metrics_df["reasoning_category"],
    category_metrics_df["partial_match_accuracy"] * 100
)
plt.ylabel("Partial Match Accuracy (%)")
plt.title("Baseline Performance by Reasoning Category")
plt.ylim(0, 100)

save_current_figure("reasoning_category_performance.png")


# ------------------------------------------------------------
# Visualization 3: Evidence Usage Distribution
# ------------------------------------------------------------

if "evidence_record_count" in evaluation_df.columns:
    plt.figure(figsize=(8, 5))
    plt.hist(
        evaluation_df["evidence_record_count"],
        bins=10
    )
    plt.xlabel("Evidence Records Used")
    plt.ylabel("Number of Samples")
    plt.title("Evidence Usage Distribution")

    save_current_figure("evidence_usage_distribution.png")


# ------------------------------------------------------------
# Visualization 4: Ground Truth vs Prediction Text Length
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))
plt.bar(
    ["Ground Truth", "Prediction"],
    [
        evaluation_df["ground_truth_word_count"].mean(),
        evaluation_df["prediction_word_count"].mean(),
    ]
)
plt.ylabel("Average Word Count")
plt.title("Average Answer Length Comparison")

save_current_figure("text_length_comparison.png")


# ------------------------------------------------------------
# Visualization 5: Runtime Summary
# ------------------------------------------------------------

runtime_plot_df = runtime_analysis_df[
    runtime_analysis_df["metric"].isin(
        [
            "elapsed_time_seconds",
            "average_time_per_sample_seconds",
        ]
    )
].copy()

runtime_plot_df["display_metric"] = (
    runtime_plot_df["metric"]
    .replace(
        {
            "elapsed_time_seconds":
                "Elapsed Runtime (sec)",
            "average_time_per_sample_seconds":
                "Avg Runtime / Sample",
        }
    )
)

plt.figure(figsize=(8, 5))
plt.bar(
    runtime_plot_df["display_metric"],
    runtime_plot_df["value"]
)
plt.ylabel("Seconds")
plt.title("Baseline Runtime Summary")

save_current_figure("runtime_comparison.png")


# ------------------------------------------------------------
# Display Generated Figure Summary
# ------------------------------------------------------------

generated_figures_df = pd.DataFrame(generated_figures)

print("Generated Figures")
print("-" * 60)

display(
    generated_figures_df.assign(
        size_kb=generated_figures_df["size_kb"].round(1)
    )[["figure", "size_kb"]]
)



### 🔷 Step 10 — Save Evaluation Results

* Save the unified evaluation dataset to the project output directory.
* Save prediction verification summaries and evaluation metric tables.
* Save reasoning-category, question-type, answer-length, runtime, and evidence utilization analyses.
* Save generated figure metadata for tracking visualization outputs.
* Preserve evaluation artifacts for later reporting, comparison, and project documentation.



In [ ]:
# ============================================================
# Step 10: Save Evaluation Results
# ============================================================

import os

print("Saving evaluation results...\n")

# ------------------------------------------------------------
# Create Output Directory
# ------------------------------------------------------------

evaluation_report_dir = (
    "outputs/evaluation/reports"
)

os.makedirs(
    evaluation_report_dir,
    exist_ok=True
)

# ------------------------------------------------------------
# Define Output Files
# ------------------------------------------------------------

evaluation_dataset_file = os.path.join(
    evaluation_report_dir,
    "evaluation_dataset.csv"
)

prediction_verification_file = os.path.join(
    evaluation_report_dir,
    "prediction_verification.csv"
)

evaluation_metrics_file = os.path.join(
    evaluation_report_dir,
    "evaluation_metrics.csv"
)

category_metrics_file = os.path.join(
    evaluation_report_dir,
    "category_metrics.csv"
)

question_type_metrics_file = os.path.join(
    evaluation_report_dir,
    "question_type_metrics.csv"
)

answer_length_metrics_file = os.path.join(
    evaluation_report_dir,
    "answer_length_metrics.csv"
)

runtime_analysis_file = os.path.join(
    evaluation_report_dir,
    "runtime_analysis.csv"
)

evidence_analysis_file = os.path.join(
    evaluation_report_dir,
    "evidence_analysis.csv"
)

generated_figures_file = os.path.join(
    evaluation_report_dir,
    "generated_figures.csv"
)

# ------------------------------------------------------------
# Save Evaluation DataFrames
# ------------------------------------------------------------

evaluation_df.to_csv(
    evaluation_dataset_file,
    index=False
)

prediction_verification_df.to_csv(
    prediction_verification_file,
    index=False
)

evaluation_metrics_df.to_csv(
    evaluation_metrics_file,
    index=False
)

category_metrics_df.to_csv(
    category_metrics_file,
    index=False
)

question_type_metrics_df.to_csv(
    question_type_metrics_file,
    index=False
)

answer_length_metrics_df.to_csv(
    answer_length_metrics_file,
    index=False
)

runtime_analysis_df.to_csv(
    runtime_analysis_file,
    index=False
)

evidence_usage_df.to_csv(
    evidence_analysis_file,
    index=False
)

generated_figures_df.to_csv(
    generated_figures_file,
    index=False
)

# ------------------------------------------------------------
# Display Saved Files
# ------------------------------------------------------------

saved_files = [
    evaluation_dataset_file,
    prediction_verification_file,
    evaluation_metrics_file,
    category_metrics_file,
    question_type_metrics_file,
    answer_length_metrics_file,
    runtime_analysis_file,
    evidence_analysis_file,
    generated_figures_file,
]

print("Saved Evaluation Files")
print("-" * 60)

for file_path in saved_files:

    file_size_kb = (
        os.path.getsize(file_path)
        / 1024
    )

    print(
        f"{os.path.basename(file_path):<35}"
        f"{file_size_kb:8.1f} KB"
    )



### 🔷 Step 11 — Display Evaluation Results

* Display overall evaluation metrics and prediction verification summaries.
* Present reasoning-category, question-type, runtime, evidence, and representation analyses.
* Display generated visualization summaries and saved figure information.
* Review key experiment findings and performance statistics.
* Confirm that evaluation outputs, reports, and visualization artifacts were successfully generated.


In [ ]:
# ============================================================
# Step 11: Display Evaluation Results
# ============================================================

import os

print("Evaluation Results")
print("=" * 60)

# ------------------------------------------------------------
# Display Overall Evaluation Metrics
# ------------------------------------------------------------

print("\nOverall Evaluation Metrics")
print("-" * 60)

display(evaluation_metrics_df)

# ------------------------------------------------------------
# Display Prediction Verification Summary
# ------------------------------------------------------------

print("\nPrediction Verification Summary")
print("-" * 60)

display(prediction_verification_df)

# ------------------------------------------------------------
# Display Reasoning Category Metrics
# ------------------------------------------------------------

print("\nReasoning Category Metrics")
print("-" * 60)

display(category_metrics_df)

# ------------------------------------------------------------
# Display Question Type Metrics
# ------------------------------------------------------------

print("\nQuestion Type Metrics")
print("-" * 60)

display(question_type_metrics_df)

# ------------------------------------------------------------
# Display Runtime Analysis
# ------------------------------------------------------------

print("\nRuntime Analysis")
print("-" * 60)

display(runtime_analysis_df)

# ------------------------------------------------------------
# Display Evidence Utilization Analysis
# ------------------------------------------------------------

print("\nEvidence Utilization Analysis")
print("-" * 60)

display(evidence_usage_df)

# ------------------------------------------------------------
# Display Generated Visualizations
# ------------------------------------------------------------

print("\nGenerated Visualizations")
print("-" * 60)

display(
    generated_figures_df.assign(
        size_kb=generated_figures_df["size_kb"].round(1)
    )
)

# ------------------------------------------------------------
# Executive Summary
# ------------------------------------------------------------

total_predictions = len(evaluation_df)

unique_videos = (
    evaluation_df["video"].nunique()
)

avg_evidence = (
    evaluation_df["evidence_record_count"].mean()
    if "evidence_record_count" in evaluation_df.columns
    else 0
)

avg_runtime = float(
    runtime_analysis_df.loc[
        runtime_analysis_df["metric"]
        == "average_time_per_sample_seconds",
        "value"
    ].iloc[0]
)

full_runtime = float(
    runtime_analysis_df.loc[
        runtime_analysis_df["metric"]
        == "projected_full_dataset_runtime_hours",
        "value"
    ].iloc[0]
)

exact_match_accuracy = float(
    evaluation_metrics_df.loc[
        evaluation_metrics_df["metric"]
        == "exact_match_accuracy",
        "value"
    ].iloc[0]
)

partial_match_accuracy = float(
    evaluation_metrics_df.loc[
        evaluation_metrics_df["metric"]
        == "partial_match_accuracy",
        "value"
    ].iloc[0]
)

print("\nExecutive Summary")
print("-" * 60)

print(
    f"Samples Evaluated                 : "
    f"{total_predictions:,}"
)

print(
    f"Unique Videos Evaluated           : "
    f"{unique_videos:,}"
)

print(
    f"Average Evidence Records/Sample   : "
    f"{avg_evidence:.2f}"
)

print(
    f"Exact Match Accuracy              : "
    f"{exact_match_accuracy:.2%}"
)

print(
    f"Partial Match Accuracy            : "
    f"{partial_match_accuracy:.2%}"
)

print(
    f"Average Runtime per Sample (sec)  : "
    f"{avg_runtime:.2f}"
)

print(
    f"Projected Full Dataset Runtime    : "
    f"{full_runtime:.2f} hours"
)

print("\nNotebook 07 evaluation complete.")

